# 99 · Comparación de resultados (segmentadores, clasificadores y pipelines)

Reúne los `resultados/*.json` que dejan los cuadernillos y los compara. **Funciona con lo que haya**: si falta un experimento, compara los disponibles, dice qué hueco queda y qué comparaciones quedan limitadas.

| Familia | Qué es | Qué se compara aquí |
|---|---|---|
| **A** | Segmentador **agnóstico** (una clase «comida») | Calidad de máscara: AP, AP50, AP75, AR100, por tamaño; eficiencia |
| **B** | Segmentador **con 323 clases** (modelo integrado) | AP con clases, exactitud de clase sobre las instancias emparejadas; AP agnóstico |
| **C** | **Clasificador** de recortes (con máscaras *reales*: techo/oráculo) | Top-1, Top-5, macro-F1, recall balanceado, IC 95 % y comparación pareada |
| **P** | **Pipeline P1 = A + C** (opcional) | AP con clases frente a B; la sección 6 lo construye solo si se pide |

**Los 10 cuadernillos** (B1 canónico: `B1_yolo26_clases_v4_final_m.ipynb`; la variante `v4` es un ensayo anterior):

| Perfil | Segmentador A | Clasificador C | Integrado B |
|---|---|---|---|
| **Ligero** | A2 · YOLO26n agnóstico | C2 · MobileNetV3 | B2 · YOLO26n con clases |
| **Robusto** | A4 · Mask2Former agnóstico | C3 · Inception-V4 | B4 · Mask2Former con clases |

A1 (YOLO26m agnóstico) y B1 (YOLO26m con clases) son variantes extra de la familia YOLO.

**Resultados de test al preparar esta versión:** A4, B4, C1, C2 y C3; **pendientes** A1, A2, B1 y B2. La sección 1.1 recalcula el estado real cada vez que se ejecuta. Los cuadernillos generan los JSON; basta con copiar `resultados/A1_yolo26m.json`, `A2_yolo26n.json`, `B1_yolo26m.json` y `B2_yolo26n.json` y volver a ejecutar. La sección 1.1 audita runs, cuadernillos y resultados, y la 5 compara los perfiles.

**Cómo se usa**
1. Ejecuta las secciones 1–5 y 7: solo leen JSON (no necesitan GPU ni datos).
2. La sección 5 compara los perfiles **por piezas** (A, C y B por separado), que es lo que se pide para la presentación. La sección 6 (P1 completos) es opcional y está apagada (`run_pipelines=False`): necesita GPU, imágenes y pesos.
3. Si añades los JSON que faltan, vuelve a ejecutar: las tablas se completan solas.

**Reglas de la comparación**
- Solo se comparan resultados con **las mismas particiones** (`splits_hash`). Si hay mezcla, se avisa y se excluyen los que no coincidan con la mayoría.
- Los **pilotos** (1 época, 5 % de datos) se ignoran en cuanto existe algún resultado real.
- A y B se evalúan contra las máscaras **originales** de COCO; los recortes de C usan máscaras refinadas con SAM 2 (cuadernillo 00). Las cifras de C son un **techo** para A+C.
- Dentro de una familia, compara solo modelos con el **mismo tamaño de entrada** (`imgsz`); la tabla de inventario lo muestra.


In [1]:
import os, sys, re, json, csv, time, glob, shutil, subprocess, importlib, textwrap, warnings
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*non-interactive.*")
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

def ensure(pkg, import_name=None):
    try: importlib.import_module(import_name or pkg)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
ensure("matplotlib")
import matplotlib.pyplot as plt
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw): return x
try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/calorie_counter")          # donde los demás cuadernillos dejan runs/, predicciones/ y resultados/
    DATA = Path("/content/data")                                    # disco local (solo la sección 6 lo usa)
elif Path("/kaggle/working").exists():
    WORK, DATA = Path("/kaggle/working/calorie_counter"), Path("/kaggle/working/data")
else:
    WORK, DATA = Path("./calorie_counter"), Path("./data")
WORK.mkdir(parents=True, exist_ok=True)
print("WORK (persistente):", WORK)

WORK (persistente): calorie_counter


In [2]:
CFG = dict(
    # --- qué resultados leer ---
    results_dirs   = None,     # None = WORK/resultados y WORK/resultados_pilot; o una lista de carpetas
    notebooks_dir  = None,     # None = buscar ../cuadernillos; ruta a la carpeta de cuadernillos para la auditoría
    include_pilot  = "auto",   # "auto": los pilotos solo se usan si NO hay ningún resultado real | True: se incluyen siempre | False: nunca
    strict_splits  = True,     # excluir los resultados cuyas particiones (splits_hash) no coinciden con la mayoría
    split          = "test",   # partición con la que se compara
    out_dir        = None,     # None = WORK/comparacion (tablas CSV/XLSX y figuras)
    min_gt_per_class = 20,     # para el AP por clase: ignorar clases con menos instancias de test
    n_boot         = 1000,     # remuestreos de la comparación pareada de clasificadores
    seed           = 42,

    # --- sección 6: pipelines P1 = segmentador agnóstico (A) + clasificador (C) ---
    run_pipelines   = False,   # True: construye y evalúa los P1 que falten (GPU + imágenes + pesos en Drive)
    rebuild_pipelines = False, # True: recalcula aunque ya exista resultados/P1_*.json
    pipeline_limit_images = None,   # int: solo las N primeras fotos (prueba rápida)
    det_min_score  = 0.05,     # el pipeline clasifica solo las detecciones con score >= este valor...
    max_det_per_img = 30,      # ...y como máximo estas por foto (limita el coste; el AP casi no cambia)
    topk           = 3,        # clases que emite cada instancia (como en COCO)
    crop_margin    = 0.02,     # MISMA regla que los recortes de entrenamiento (cuadernillo 00): caja de la máscara + 2 %
    crop_max_side  = 256,
    mask_erode_px  = 1,
    cls_batch      = 128,
    match_score_thr = 0.3,     # umbral de score para la exactitud sobre instancias emparejadas
    eval_folds     = 5,
)
OUT = Path(CFG["out_dir"]) if CFG["out_dir"] else WORK / "comparacion"
OUT.mkdir(parents=True, exist_ok=True)
SPLIT = CFG["split"]
print("Se comparará con la partición:", SPLIT, "| salidas en:", OUT)

Se comparará con la partición: test | salidas en: calorie_counter\comparacion


## 1. Inventario de resultados y cobertura

Qué hay en Drive, qué falta y si los resultados son comparables (mismas particiones y tamaños de entrada).

In [3]:
def _get(d, *ks, default=None):
    for k in ks:
        if not isinstance(d, dict) or k not in d: return default
        d = d[k]
    return d

def _num(x):
    try: return float(x) if x is not None else np.nan
    except (TypeError, ValueError): return np.nan

RESULT_DIRS = [Path(p) for p in (CFG["results_dirs"] or [WORK / "resultados", WORK / "resultados_pilot"])]

def read_results(dirs):
    """Lee todos los *.json válidos. Tolerante: un archivo roto no detiene la comparación."""
    recs, bad = {}, []
    for d in dirs:
        if not Path(d).exists(): continue
        for f in sorted(Path(d).glob("*.json")):
            try:
                r = json.loads(f.read_text(encoding="utf-8"))
                for k in ("exp_id", "family", "metrics"): assert k in r, f"falta la clave '{k}'"
                assert isinstance(r["metrics"], dict) and r["metrics"], "metrics vacío"
                assert r["family"] in "ABCP", f"familia desconocida {r['family']!r}"
            except Exception as e:
                bad.append((f.name, f"{type(e).__name__}: {e}")); continue
            r["_file"] = str(f); r["_pilot"] = bool(_get(r, "cfg", "pilot") or _get(r, "train", "pilot"))
            recs[r["exp_id"]] = r
    return recs, bad

def select_results(alld, verbose=True):
    real = {k: v for k, v in alld.items() if not v["_pilot"]}
    pil = {k: v for k, v in alld.items() if v["_pilot"]}
    if not alld: use, why = {}, "no hay resultados"
    elif CFG["include_pilot"] is True: use, why = dict(alld), "include_pilot=True"
    elif CFG["include_pilot"] is False or real: use, why = dict(real), ("hay resultados reales" if real else "include_pilot=False")
    else: use, why = dict(pil), "no hay resultados reales: se usan los pilotos (solo sirven para comprobar el flujo)"
    ignored = [k for k in alld if k not in use]
    if verbose:
        print(f"Resultados leídos: {len(alld)} ({len(real)} reales, {len(pil)} piloto) | usados: {len(use)} [{why}]")
        if ignored: print("  Ignorados (piloto):", ", ".join(sorted(ignored)))
    # particiones distintas -> comparación inválida
    hashes = Counter(_get(r, "data", "splits_hash") for r in use.values() if _get(r, "data", "splits_hash"))
    if len(hashes) > 1:
        top = hashes.most_common(1)[0][0]
        odd = [k for k, r in use.items() if _get(r, "data", "splits_hash") not in (None, top)]
        print(f"⚠️ Hay {len(hashes)} particiones distintas (splits_hash). Mayoritaria: {top[:10]}…  Distintas: {odd}")
        if CFG["strict_splits"]:
            for k in odd: use.pop(k)
            print("  → excluidos de la comparación (strict_splits=True).")
    return use

ALL, BAD_FILES = read_results(RESULT_DIRS)
R = select_results(ALL)
for n, e in BAD_FILES: print(f"⚠️ {n} no es un resultado válido: {e}")
if not R:
    print("\nTodavía no hay resultados que comparar. Ejecuta antes los cuadernillos A/B/C (cada uno escribe resultados/<id>.json) y vuelve aquí.")

Resultados leídos: 13 (9 reales, 4 piloto) | usados: 9 [hay resultados reales]
  Ignorados (piloto): A4_mask2former_swin_tiny_pilot, B4_mask2former_swin_tiny_pilot, C2_mobilenetv3_large_100_pilot, C3_inception_v4_pilot


In [4]:
EXPECTED = [f"{f}{i}" for f in "ABC" for i in (1, 2, 3, 4)]
def slot_of(eid):
    m = re.match(r"^([ABCP]\d+)", eid); return m.group(1) if m else eid

def inventory():
    rows = []
    by_slot = defaultdict(list)
    for eid in ALL: by_slot[slot_of(eid)].append(eid)
    slots = EXPECTED + sorted(s for s in by_slot if s not in EXPECTED)
    for s in slots:
        ids = sorted(by_slot.get(s, []))
        used = [i for i in ids if i in R]
        if used: estado = "✅ real" if not any(R[i]["_pilot"] for i in used) else "🧪 piloto"
        elif ids: estado = "🧪 solo piloto (no se compara)" if all(ALL[i]["_pilot"] for i in ids) else "⚠️ excluido"
        else: estado = "❌ falta"
        for i in (used or ids or [None]):
            r = ALL.get(i) if i else None
            rows.append(dict(slot=s, estado=estado, exp_id=i, modelo=_get(r, "model"), imgsz=_get(r, "cfg", "imgsz") or _get(r, "efficiency", "imgsz"),
                             épocas=_get(r, "train", "epochs_run"), creado=(_get(r, "created") or "")[:16]))
    return pd.DataFrame(rows)

INV = inventory(); display(INV)
falt = INV[INV.estado.str.startswith("❌")].slot.tolist()
print(f"\nCobertura: {INV[INV.estado.str.startswith(('✅','🧪 piloto'))].slot.nunique()} de {len(EXPECTED)} experimentos esperados con resultados."
      + (f" Faltan: {', '.join(falt)}." if falt else ""))
# tamaños de entrada: comparar solo dentro de la misma resolución
for fam in "ABC":
    sz = sorted({str(x) for x in INV[(INV.slot.str.startswith(fam)) & INV.exp_id.notna()].imgsz.dropna()})
    if len(sz) > 1: print(f"⚠️ Familia {fam}: hay varios tamaños de entrada {sz}; la comparación entre ellos mezcla modelo y resolución.")

,slot,estado,exp_id,modelo,imgsz,épocas,creado
0,A1,✅ real,A1_yolo26m,YOLO26m-seg,640.0,40.0,2026-11-06T03:11
1,A2,✅ real,A2_yolo26n,YOLO26n-seg,224.0,40.0,2026-11-06T03:34
2,A3,❌ falta,NaN,NaN,NaN,NaN,
3,A4,✅ real,A4_mask2former_swin_tiny,Mask2Former Swin-tiny,640.0,6.0,2026-10-31T14:17
4,B1,✅ real,B1_yolo26m,YOLO26m-seg,640.0,40.0,2026-11-06T03:09
5,B2,✅ real,B2_yolo26n,YOLO26n-seg,224.0,40.0,2026-11-06T03:25
6,B3,❌ falta,NaN,NaN,NaN,NaN,
7,B4,✅ real,B4_mask2former_swin_tiny,Mask2Former Swin-tiny,224.0,20.0,2026-11-03T08:08
8,C1,✅ real,C1_cnn_propia,"FoodNet (CNN propia, desde cero)",160.0,40.0,2026-10-29T19:18
9,C2,✅ real,C2_mobilenetv3_large_100,timm mobilenetv3_large_100.ra4_e3600_r224_in1k,224.0,15.0,2026-11-01T23:01



Cobertura: 9 de 12 experimentos esperados con resultados. Faltan: A3, B3, C4.
⚠️ Familia A: hay varios tamaños de entrada ['224.0', '640.0']; la comparación entre ellos mezcla modelo y resolución.
⚠️ Familia B: hay varios tamaños de entrada ['224.0', '640.0']; la comparación entre ellos mezcla modelo y resolución.
⚠️ Familia C: hay varios tamaños de entrada ['160.0', '224.0']; la comparación entre ellos mezcla modelo y resolución.


### 1.1 Cuadernillos, runs y resultados: auditoría

Cruza cada carpeta de `runs/` con su cuadernillo y con el JSON de resultados. Sirve para responder tres preguntas: qué entrenamientos están completos, cuáles no dejaron resultado de test y si hay algún modelo con pesos completos **sin cuadernillo**.

In [5]:
# Rutas de los cuadernillos (en Colab puede que no estén en Drive; si no aparecen, se omite esa columna).
NB_DIR = Path(CFG["notebooks_dir"]) if CFG.get("notebooks_dir") else None
if NB_DIR is None:
    for cand in (WORK.parent / "cuadernillos", WORK / "cuadernillos", Path.cwd() / "cuadernillos"):
        if cand.exists():
            NB_DIR = cand
            break

NOTEBOOK_OF = {
    "A1": "A1_yolo26_agnostico_v2_m.ipynb",
    "A2": "A2_yolo26n_agnostico_corregido.ipynb",
    "A4": "A4_mask2former_agnostico_opt.ipynb",
    "B1": "B1_yolo26_clases_v4_final_m.ipynb",   # versión canónica
    "B2": "B2_yolo26n_clases_corregido.ipynb",
    "B4": "B4_mask2former_clases_FIX.ipynb",
    "C1": "C1_cnn_propia_b_c.ipynb",
    "C2": "C2_mobilenetv3_224.ipynb",
    "C3": "C3_inception_v_unificado.ipynb",
}
EXPECTED_JSON = {
    "A1": "A1_yolo26m.json", "A2": "A2_yolo26n.json",
    "B1": "B1_yolo26m.json", "B2": "B2_yolo26n.json",
}

def _last_epoch_csv(p):
    if not Path(p).exists():
        return None
    try:
        with open(p, encoding="utf-8", errors="ignore") as fh:
            rows = list(csv.DictReader(fh))
        return int(float(rows[-1]["epoch"])) if rows else None
    except Exception:
        return None

def _planned_epochs(run_dir):
    cfg = run_dir / "config.json"
    if cfg.exists():
        try:
            return json.loads(cfg.read_text(encoding="utf-8")).get("epochs")
        except Exception:
            pass
    args = run_dir / "args.yaml"
    if args.exists():
        m = re.search(r"^epochs:\s*(\d+)", args.read_text(encoding="utf-8", errors="ignore"), re.M)
        if m:
            return int(m.group(1))
    return None

def audit_runs():
    rows = []
    runs_dir = WORK / "runs"
    if not runs_dir.exists():
        return pd.DataFrame()
    for d in sorted(p for p in runs_dir.iterdir() if p.is_dir()):
        name = d.name
        best = next((p for p in (d / "weights" / "best.pt", d / "best.pt") if p.exists()), None)
        done_file = d / "DONE.json"
        done = json.loads(done_file.read_text(encoding="utf-8")) if done_file.exists() else {}
        epochs = done.get("epochs_run") or _last_epoch_csv(d / "history.csv") or _last_epoch_csv(d / "results.csv")
        planned = _planned_epochs(d)
        if planned is None and done.get("finished"):
            planned = epochs   # sin args.yaml/config.json: el DONE confirma que terminó
        slot = re.match(r"^([ABCP]\d+)", name)
        slot = slot.group(1) if slot else name
        pilot = name.endswith("_pilot")
        has_nb = bool(NB_DIR and (NB_DIR / NOTEBOOK_OF[slot]).exists()) if slot in NOTEBOOK_OF else False
        result = (WORK / "resultados" / f"{name}.json").exists()
        if best is None:
            state = "sin pesos"
        elif pilot:
            state = "piloto"
        elif done.get("finished") or (epochs and planned and epochs >= planned):
            state = "completo con resultado" if result else "completo sin resultado"
        else:
            state = "incompleto"
        if pilot:
            nb_cell = "—"
        elif NB_DIR is None:
            nb_cell = "?"          # no se encontró la carpeta de cuadernillos (p. ej. en Colab sin Drive)
        elif has_nb:
            nb_cell = NOTEBOOK_OF[slot]
        else:
            nb_cell = "falta"
        rows.append(dict(
            slot=slot, run=name,
            pesos_M=round(best.stat().st_size / 1e6, 1) if best else np.nan,
            epocas=epochs, planeadas=planned, DONE="sí" if done else "no",
            cuadernillo=nb_cell, resultado="sí" if result else "no", estado=state,
        ))
    return pd.DataFrame(rows)

AUDIT = audit_runs()
if AUDIT.empty:
    print("No se encontró runs/ en", WORK)
else:
    display(AUDIT)
    AUDIT.to_csv(OUT / "auditoria_runs.csv", index=False, encoding="utf-8-sig")
    sin_resultado = AUDIT[AUDIT.estado == "completo sin resultado"].run.tolist()
    incompletos = AUDIT[AUDIT.estado == "incompleto"].run.tolist()
    sin_pesos = AUDIT[AUDIT.estado == "sin pesos"].run.tolist()
    print("\nEntrenamiento completo pero SIN resultado de test:", ", ".join(sin_resultado) or "ninguno")
    print("Entrenamientos incompletos:", ", ".join(incompletos) or "ninguno")
    for _, r in AUDIT[AUDIT.estado == "incompleto"].iterrows():
        ep = int(r.epocas) if pd.notna(r.epocas) else "?"
        pl = int(r.planeadas) if pd.notna(r.planeadas) else "?"
        print(f"  • {r.run}: {ep}/{pl} épocas, DONE={r.DONE} → conviene terminarlo o reentrenar.")
    print("Runs sin pesos (no evaluables):", ", ".join(sin_pesos) or "ninguno")
    if NB_DIR is None:
        print("Carpeta de cuadernillos no encontrada: no se puede comprobar si falta algún cuadernillo.")
    else:
        sin_nb = AUDIT[(AUDIT.pesos_M.notna()) & (AUDIT.estado != "piloto") & (AUDIT.cuadernillo == "falta")].run.tolist()
        print("Con pesos y SIN cuadernillo:", ", ".join(sin_nb) or "ninguno")
    faltan = [f"resultados/{EXPECTED_JSON[s]}" for s in ("A1", "A2", "B1", "B2")
              if s in AUDIT.slot.values and not (WORK / "resultados" / EXPECTED_JSON[s]).exists()]
    if faltan:
        print("\nJSON de test que faltan (cópialos a la carpeta resultados/):")
        for f in faltan:
            print("  •", f)


,slot,run,pesos_M,epocas,planeadas,DONE,cuadernillo,resultado,estado
0,A1,A1_yolo26m,54.5,40.0,40,sí,A1_yolo26_agnostico_v2_m.ipynb,sí,completo con resultado
1,A2,A2_yolo26n,6.6,40.0,40,sí,A2_yolo26n_agnostico_corregido.ipynb,sí,completo con resultado
2,A4,A4_mask2former_swin_tiny,189.9,6.0,6,sí,A4_mask2former_agnostico_opt.ipynb,sí,completo con resultado
3,A4,A4_mask2former_swin_tiny_pilot,189.9,1.0,6,sí,—,no,piloto
4,B1,B1_yolo26m,166.0,39.0,40,no,B1_yolo26_clases_v4_final_m.ipynb,sí,incompleto
5,B2,B2_yolo26n,6.9,40.0,40,sí,B2_yolo26n_clases_corregido.ipynb,sí,completo con resultado
6,B4,B4_mask2former_swin_tiny,190.2,20.0,20,sí,B4_mask2former_clases_FIX.ipynb,sí,completo con resultado
7,B4,B4_mask2former_swin_tiny_pilot,190.2,1.0,1,sí,—,no,piloto
8,C1,C1_cnn_propia,45.9,40.0,40,sí,C1_cnn_propia_b_c.ipynb,sí,completo con resultado
9,C2,C2_mobilenetv3_large_100,18.7,15.0,30,sí,C2_mobilenetv3_224.ipynb,sí,completo con resultado



Entrenamiento completo pero SIN resultado de test: ninguno
Entrenamientos incompletos: B1_yolo26m
  • B1_yolo26m: 39/40 épocas, DONE=no → conviene terminarlo o reentrenar.
Runs sin pesos (no evaluables): C2_mobilenetv5_300m_pilot
Con pesos y SIN cuadernillo: ninguno


## 2. Segmentadores: calidad de la máscara (modo agnóstico)

Compara A (segmentadores agnósticos) y, como referencia, los B evaluados en modo agnóstico (ignoran la clase). **Cómo leerlo:**
- **AP50 y AR100** dicen si el modelo *encuentra* los alimentos y los separa con un solape razonable; son las cifras más robustas al ruido de los polígonos originales y las que más importan para el pipeline.
- **AP75 y AP (0.5:0.95)** miden cuánto se ajusta el contorno **al polígono original**, que es tosco: sirven para ordenar modelos, pero un modelo con borde más fiel a la comida real puede puntuar igual o peor. No las leas como «calidad real del borde».
- La barra de error es la desviación del AP entre los trozos de imágenes (`n_folds`), una idea de la incertidumbre, no un intervalo de confianza.

In [6]:
def eff_cols(r):
    eff = r.get("efficiency") or {}; lat = eff.get("latency") or {}
    return dict(params_M=_num(eff.get("params")) / 1e6, lat_ms=_num(lat.get("mean_ms")), p95_ms=_num(lat.get("p95_ms")),
                gpu_MB=_num(eff.get("gpu_mem_mb")), épocas=_get(r, "train", "epochs_run"), entreno_min=_num(_get(r, "train", "time_s")) / 60,
                imgsz=_get(r, "cfg", "imgsz") or eff.get("imgsz"), piloto=r["_pilot"])

def seg_table(mode, fams=("A", "B")):
    rows = []
    for eid, r in sorted(R.items()):
        if r["family"] not in fams: continue
        m = _get(r, "metrics", SPLIT, mode)
        if not isinstance(m, dict) or m.get("AP") is None and m.get("AP50") is None: continue
        d = dict(exp_id=eid, fam=r["family"], modelo=r.get("model"))
        for k in ("AP", "AP50", "AP75", "APs", "APm", "APl", "AR100"): d[k] = _num(m.get(k))
        d["AP_fold_mean"], d["AP_fold_std"] = _num(m.get("AP_fold_mean")), _num(m.get("AP_fold_std"))
        d["n_img"], d["n_gt"] = m.get("n_images"), m.get("n_gt")
        d.update(eff_cols(r)); rows.append(d)
    return pd.DataFrame(rows)

def save_df(df, name):
    if df is None or df.empty: return
    df.to_csv(OUT / f"{name}.csv", index=False, encoding="utf-8-sig")

def save_fig(name):
    plt.savefig(OUT / f"{name}.png", dpi=140, bbox_inches="tight")

def bar_compare(df, cols, title, err=None, label="exp_id"):
    d = df.dropna(subset=cols, how="all")
    if d.empty: return
    n, w = len(cols), 0.8 / len(cols)
    fig, ax = plt.subplots(figsize=(max(6, 1.3 * len(d) + 2), 3.8))
    for j, c in enumerate(cols):
        e = d[err[c]].fillna(0).values if err and c in err else None
        ax.bar(np.arange(len(d)) + j * w, d[c].fillna(0).values, w, yerr=e, capsize=3, label=c)
    ax.set_xticks(np.arange(len(d)) + w * (n - 1) / 2); ax.set_xticklabels(d[label], rotation=30, ha="right", fontsize=8)
    ax.set_title(title); ax.legend(fontsize=8); ax.set_ylim(0, 1); ax.grid(axis="y", alpha=.3)
    plt.tight_layout(); return fig

def scatter_eff(df, y, title, size_col="params_M"):
    d = df.dropna(subset=["lat_ms", y])
    if d.empty: return
    fig, ax = plt.subplots(figsize=(6, 4))
    s = 40 + 12 * d[size_col].fillna(1).clip(upper=100).values
    ax.scatter(d["lat_ms"], d[y], s=s, alpha=.6)
    for _, row in d.iterrows(): ax.annotate(row["exp_id"], (row["lat_ms"], row[y]), fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("latencia (ms / imagen o recorte)"); ax.set_ylabel(y); ax.set_title(title + "  (tamaño = parámetros)"); ax.grid(alpha=.3)
    plt.tight_layout(); return fig

SEG_AG = seg_table("agnostic")
if SEG_AG.empty:
    print("No hay resultados de segmentadores (A o B) todavía.")
else:
    cols = ["exp_id", "fam", "AP50", "AR100", "AP", "AP75", "APs", "APm", "APl", "AP_fold_mean", "AP_fold_std", "params_M", "lat_ms", "p95_ms", "gpu_MB", "épocas", "entreno_min", "imgsz", "piloto"]
    display(SEG_AG.sort_values("AP50", ascending=False)[cols].round(3).reset_index(drop=True)); save_df(SEG_AG, "segmentadores_agnostico")
    f = bar_compare(SEG_AG.sort_values("AP50", ascending=False), ["AP50", "AP", "AR100"], f"Segmentación agnóstica ({SPLIT})", err={"AP": "AP_fold_std"})
    if f is not None: save_fig("seg_agnostico_barras"); plt.show()
    f = scatter_eff(SEG_AG, "AP50", "Calidad frente a velocidad (segmentadores)")
    if f is not None: save_fig("seg_agnostico_eficiencia"); plt.show()
    if len(SEG_AG) == 1: print("ℹ️ Solo hay un segmentador: no hay con qué compararlo todavía.")

,exp_id,fam,AP50,AR100,AP,AP75,APs,APm,APl,AP_fold_mean,AP_fold_std,params_M,lat_ms,p95_ms,gpu_MB,épocas,entreno_min,imgsz,piloto
0,A4_mask2former_swin_tiny,A,0.752,0.726,0.548,0.597,0.083,0.224,0.580,0.549,0.021,47.403,111.286,115.208,701.394,6,579.998,640,False
1,A1_yolo26m,A,0.682,0.551,0.475,0.518,0.011,0.214,0.500,0.477,0.021,26.970,20.803,21.609,235.494,40,143.140,640,False
2,B1_yolo26m,B,0.670,0.567,0.433,0.470,0.027,0.190,0.456,0.434,0.016,27.549,19.869,20.933,217.610,40,22.348,640,False
3,B4_mask2former_swin_tiny,B,0.657,0.709,0.454,0.500,0.067,0.174,0.483,0.455,0.015,47.486,74.176,74.880,475.892,20,575.235,224,False
4,A2_yolo26n,A,0.651,0.535,0.445,0.486,0.016,0.174,0.472,0.447,0.021,3.083,15.987,16.806,142.643,40,482.068,224,False
5,B2_yolo26n,B,0.601,0.553,0.387,0.422,0.041,0.146,0.411,0.388,0.019,3.260,15.765,16.128,116.478,40,479.017,224,False


## 3. Segmentadores con clases (familia B) y exactitud de clase

El **AP con clases** exige que máscara *y* clase coincidan. Además, `matched_cls` mide la exactitud de la clase **solo sobre las instancias que el modelo encontró** (emparejadas por máscara con IoU ≥ 0.5, sin mirar la clase): separa «¿encontró el alimento?» (`det_recall`) de «¿le puso la clase correcta?» (`acc_clase`).

In [7]:
def seg_cls_table(fams=("B", "P")):
    rows = []
    for eid, r in sorted(R.items()):
        if r["family"] not in fams: continue
        m = _get(r, "metrics", SPLIT, "class_aware")
        if not isinstance(m, dict): continue
        mc = _get(r, "metrics", SPLIT, "matched_cls") or {}
        d = dict(exp_id=eid, fam=r["family"], modelo=r.get("model"), AP=_num(m.get("AP")), AP50=_num(m.get("AP50")), AP75=_num(m.get("AP75")), AR100=_num(m.get("AR100")),
                 AP_fold_std=_num(m.get("AP_fold_std")), AP50_agn=_num(_get(r, "metrics", SPLIT, "agnostic", "AP50")),
                 det_recall=_num(mc.get("det_recall")), det_precision=_num(mc.get("det_precision")), acc_clase=_num(mc.get("accuracy")),
                 macroF1_clase=_num(mc.get("macro_f1")), bal_acc_clase=_num(mc.get("balanced_acc")), n_emparejadas=mc.get("n_matched"))
        d.update(eff_cols(r)); rows.append(d)
    return pd.DataFrame(rows)

SEG_CL = seg_cls_table(("B",))
if SEG_CL.empty:
    print("No hay resultados de segmentadores con clases (B) todavía.")
else:
    cols = ["exp_id", "AP50", "AR100", "AP", "AP75", "AP50_agn", "det_recall", "det_precision", "acc_clase", "macroF1_clase", "bal_acc_clase", "params_M", "lat_ms", "gpu_MB", "épocas", "imgsz", "piloto"]
    display(SEG_CL.sort_values("AP50", ascending=False)[cols].round(3).reset_index(drop=True)); save_df(SEG_CL, "segmentadores_clases")
    f = bar_compare(SEG_CL.sort_values("AP50", ascending=False), ["AP50", "AR100", "acc_clase"], f"Segmentadores con clases ({SPLIT})")
    if f is not None: save_fig("seg_clases_barras"); plt.show()
    best = SEG_CL.sort_values("AP50", ascending=False).iloc[0]["exp_id"]
    pc = _get(R[best], "metrics", SPLIT, "class_aware", "per_class")
    if pc:
        d = pd.DataFrame(pc); d = d[(d.n_gt >= CFG["min_gt_per_class"]) & d.AP.notna()].sort_values("AP", ascending=False)
        print(f"\nAP por clase de {best} (clases con ≥ {CFG['min_gt_per_class']} instancias de {SPLIT}; {len(d)} clases)")
        display(pd.concat([d.head(8).assign(grupo="mejores"), d.tail(8).assign(grupo="peores")])[["grupo", "name", "AP", "AP50", "n_gt"]].round(3).reset_index(drop=True))

,exp_id,AP50,AR100,AP,AP75,AP50_agn,det_recall,det_precision,acc_clase,macroF1_clase,bal_acc_clase,params_M,lat_ms,gpu_MB,épocas,imgsz,piloto
0,B4_mask2former_swin_tiny,0.352,0.605,0.265,0.288,0.657,NaN,NaN,NaN,NaN,NaN,47.486,74.176,475.892,20,224,False
1,B1_yolo26m,0.325,0.326,0.232,0.255,0.670,0.585,0.790,0.736,0.542,0.547,27.549,19.869,217.610,40,640,False
2,B2_yolo26n,0.207,0.236,0.148,0.164,0.601,0.441,0.794,0.674,0.386,0.395,3.260,15.765,116.478,40,224,False



AP por clase de B4_mask2former_swin_tiny (clases con ≥ 20 instancias de test; 99 clases)


,grupo,name,AP,AP50,n_gt
0,mejores,croissant,0.737,0.786,23
1,mejores,apple,0.686,0.771,98
2,mejores,chips_french_fries,0.680,0.921,36
3,mejores,wine_red,0.678,0.856,81
4,mejores,pasta_spaghetti,0.655,0.828,40
5,mejores,wine_white,0.646,0.854,47
6,mejores,soup,0.645,0.676,23
7,mejores,tea,0.615,0.742,154
8,peores,tofu,0.069,0.121,25
9,peores,mushroom,0.068,0.192,34


## 4. Clasificadores (familia C)

Se evalúan sobre recortes con la **máscara real** (refinada): es el **techo** de lo que puede dar el clasificador cuando la segmentación es perfecta. Con 323 clases y mucho desbalance, el **macro-F1 y el recall balanceado** son más exigentes que el top-1 (que premia a las clases frecuentes); la brecha entre ambos indica cuánto falla la cola de clases raras. Los intervalos son bootstrap al 95 % sobre los recortes de test.

In [8]:
def cls_table():
    rows = []
    for eid, r in sorted(R.items()):
        if r["family"] != "C": continue
        m = _get(r, "metrics", SPLIT)
        if not isinstance(m, dict) or "top1" not in m: continue
        ci = m.get("bootstrap_ci") or {}
        d = dict(exp_id=eid, modelo=r.get("model"), top1=_num(m.get("top1")), top5=_num(m.get("top5")), macro_f1=_num(m.get("macro_f1")),
                 bal_acc=_num(m.get("balanced_acc")), n=m.get("n"), clases_eval=m.get("n_classes_eval"),
                 top1_lo=_num(_get(ci, "top1", "lo")), top1_hi=_num(_get(ci, "top1", "hi")),
                 f1_lo=_num(_get(ci, "macro_f1", "lo")), f1_hi=_num(_get(ci, "macro_f1", "hi")),
                 preentrenado=_get(r, "train", "pretrained"), mejor_época=_get(r, "train", "best_epoch"))
        d.update(eff_cols(r)); rows.append(d)
    return pd.DataFrame(rows)

CLS = cls_table()
if CLS.empty:
    print("No hay resultados de clasificadores (C) todavía.")
else:
    CLS["err_top1"] = (CLS.top1_hi - CLS.top1_lo) / 2; CLS["err_f1"] = (CLS.f1_hi - CLS.f1_lo) / 2
    cols = ["exp_id", "top1", "top1_lo", "top1_hi", "top5", "macro_f1", "f1_lo", "f1_hi", "bal_acc", "n", "params_M", "lat_ms", "gpu_MB", "épocas", "mejor_época", "entreno_min", "imgsz", "piloto"]
    display(CLS.sort_values("top1", ascending=False)[cols].round(3).reset_index(drop=True)); save_df(CLS, "clasificadores")
    f = bar_compare(CLS.sort_values("top1", ascending=False), ["top1", "top5", "macro_f1"], f"Clasificadores con máscara real ({SPLIT})", err={"top1": "err_top1", "macro_f1": "err_f1"})
    if f is not None: save_fig("clasificadores_barras"); plt.show()
    f = scatter_eff(CLS, "top1", "Exactitud frente a velocidad (clasificadores)")
    if f is not None: save_fig("clasificadores_eficiencia"); plt.show()
    if len(CLS) == 1: print("ℹ️ Solo hay un clasificador: no hay con qué compararlo todavía.")

,exp_id,top1,top1_lo,top1_hi,top5,macro_f1,f1_lo,f1_hi,bal_acc,n,params_M,lat_ms,gpu_MB,épocas,mejor_época,entreno_min,imgsz,piloto
0,C2_mobilenetv3_large_100,0.579,0.567,0.590,0.840,0.423,0.398,0.429,0.411,7301,4.616,10.695,599.106,15,10,79.203,224,False
1,C3_inception_v4,0.578,0.566,0.590,0.835,0.427,0.402,0.435,0.419,7301,41.639,30.553,722.760,17,11,742.713,224,False
2,C1_cnn_propia,0.506,0.493,0.517,0.776,0.308,0.289,0.315,0.300,7301,11.442,6.875,507.422,40,36,783.448,160,False


In [9]:
# Comparación PAREADA de clasificadores: mismos recortes de test, diferencia de top-1 con IC bootstrap
def load_probs(rec):
    cands = []
    p = _get(rec, "files", "probs_" + SPLIT)
    if p: cands += [Path(p), WORK / "predicciones" / Path(p).name, WORK / "predicciones_pilot" / Path(p).name]
    cands += [WORK / "predicciones" / f"{rec['exp_id']}_{SPLIT}.npz", WORK / "predicciones_pilot" / f"{rec['exp_id']}_{SPLIT}.npz"]
    for c in cands:
        if c.exists():
            z = np.load(c); return dict(ids=z["ann_ids"], y=z["y_true"], p=z["probs"].astype(np.float32))
    return None

PROBS = {}
if not CLS.empty:
    for eid in CLS.exp_id:
        pr = load_probs(R[eid])
        if pr is not None: PROBS[eid] = pr
        else: print(f"  (sin probabilidades de {eid}: no entra en la comparación pareada)")
if len(PROBS) < 2:
    print("Comparación pareada: se necesitan al menos 2 clasificadores con sus probabilidades (.npz) en Drive.")
else:
    common = sorted(set.intersection(*[set(map(int, v["ids"])) for v in PROBS.values()]))
    print(f"Recortes comunes a todos: {len(common):,}")
    HIT = {}
    for eid, v in PROBS.items():
        pos = {int(a): i for i, a in enumerate(v["ids"])}; ix = np.array([pos[a] for a in common])
        HIT[eid] = (v["p"][ix].argmax(1) == v["y"][ix])
    rng = np.random.default_rng(CFG["seed"]); N = len(common)
    boots = rng.integers(0, N, (CFG["n_boot"], N)) if N * CFG["n_boot"] <= 2e8 else None
    names = list(HIT); rows = []
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            d = HIT[a].astype(np.float32) - HIT[b].astype(np.float32)
            if boots is not None: dist = d[boots].mean(1)
            else: dist = np.array([d[rng.integers(0, N, N)].mean() for _ in range(CFG["n_boot"])])
            lo, hi = np.percentile(dist, [2.5, 97.5])
            rows.append(dict(A=a, B=b, dif_top1=d.mean(), lo=lo, hi=hi, veredicto=("A > B" if lo > 0 else "B > A" if hi < 0 else "sin diferencia clara")))
    PAIRS = pd.DataFrame(rows); display(PAIRS.round(4)); save_df(PAIRS, "clasificadores_pareado")
    print("«sin diferencia clara» = el IC 95 % de la diferencia de top-1 incluye 0.")

Recortes comunes a todos: 7,301


,A,B,dif_top1,lo,hi,veredicto
0,C1_cnn_propia,C2_mobilenetv3_large_100,-0.0727,-0.0844,-0.0620,B > A
1,C1_cnn_propia,C3_inception_v4,-0.0716,-0.0827,-0.0600,B > A
2,C2_mobilenetv3_large_100,C3_inception_v4,0.0011,-0.0090,0.0118,sin diferencia clara


«sin diferencia clara» = el IC 95 % de la diferencia de top-1 incluye 0.


## 5. Comparativas por perfiles: ligero frente a robusto (por piezas)

Para la presentación se comparan dos perfiles completos, cada uno con las dos propuestas:

| Perfil | Pipeline A (piezas A + C) | Pipeline B (integrado) | Entrada |
|---|---|---|---|
| **Ligero** | A2 (YOLO26n agnóstico) + C2 (MobileNetV3) | B2 (YOLO26n con clases) | 224 px |
| **Robusto** | A4 (Mask2Former agnóstico, 640 px) + C3 (Inception-V4, 224 px) | B4 (Mask2Former con clases, 224 px) | mixta |

Como el pipeline P1 (A+C end-to-end) no se construye aquí, la comparación es **por piezas**, que es lo pedido: se enfrentan la calidad de máscara de A, el techo de clasificación de C y el resultado integrado de B. Para A+C se muestra la **suma de parámetros y latencias** (coste estimado de las piezas, no una latencia end-to-end) y el **techo de clasificación** (top-1 de C con máscara real).

**Cómo leerlo sin engañarse**

- `AP50_A` es **agnóstico** (una sola clase «comida»): mide solo la máscara.
- Para comparar máscaras en igualdad de condiciones, usa `AP50_agn_B` (B evaluado **ignorando la clase**) frente a `AP50_A`. `AP50_B` es **con 323 clases** y añade el error de clasificación; su diferencia con A no aísla la segmentación.
- `top1_C` usa **máscaras reales**: es el **techo** del clasificador, no el resultado de P1 con máscaras predichas.
- En el perfil robusto, A4 evalúa a **640 px** y B4 a **224 px**; la latencia de A+C y la de B (y el AP) no son directamente comparables. En el ligero, A2 y B2 comparten 224 px.
- B4 no guardó `matched_cls` (exactitud de clase sobre instancias emparejadas): la comparación fina B frente al techo de C queda pendiente de recalcular ese bloque.

In [10]:
PERFILES = {
    "ligero":  dict(A="A2", C="C2", B="B2"),
    "robusto": dict(A="A4", C="C3", B="B4"),
}

def find_exp(prefix):
    ids = [e for e in ALL if e.startswith(prefix + "_")]
    real = [e for e in ids if not ALL[e]["_pilot"]]
    return sorted(real)[0] if real else None

def _piece(perfil, rol, prefix):
    eid = find_exp(prefix)
    if eid is None or eid not in R:
        return dict(perfil=perfil, rol=rol, exp_id=f"{prefix}_*", modelo=None, estado="falta resultado de test")
    r = R[eid]
    m = _get(r, "metrics", SPLIT) or {}
    row = dict(perfil=perfil, rol=rol, exp_id=eid, modelo=r.get("model"), estado="ok")
    row.update(eff_cols(r))
    if r["family"] == "A":
        ag = m.get("agnostic") or {}
        row.update(AP50=_num(ag.get("AP50")), AP=_num(ag.get("AP")), AR100=_num(ag.get("AR100")))
    elif r["family"] == "C":
        row.update(top1=_num(m.get("top1")), top5=_num(m.get("top5")), macro_f1=_num(m.get("macro_f1")))
    elif r["family"] == "B":
        ca = m.get("class_aware") or {}
        mc = m.get("matched_cls") or {}
        ag = m.get("agnostic") or {}
        row.update(AP50=_num(ca.get("AP50")), AP=_num(ca.get("AP")), AR100=_num(ca.get("AR100")),
                   AP50_agn=_num(ag.get("AP50")), AR100_agn=_num(ag.get("AR100")),
                   acc_clase=_num(mc.get("accuracy")), det_recall=_num(mc.get("det_recall")))
    return row

rows = []
for perfil, cfg in PERFILES.items():
    rows.append(_piece(perfil, "Segmentador A (agnóstico)", cfg["A"]))
    rows.append(_piece(perfil, "Clasificador C (máscara real)", cfg["C"]))
    rows.append(_piece(perfil, "Integrado B (con clases)", cfg["B"]))
PIECES = pd.DataFrame(rows)
cols = ["perfil", "rol", "exp_id", "estado", "AP50", "AP", "AR100", "AP50_agn", "AR100_agn",
        "top1", "top5", "macro_f1", "acc_clase", "det_recall", "params_M", "lat_ms", "imgsz", "épocas", "piloto"]
cols = [c for c in cols if c in PIECES.columns]
display(PIECES[cols].round(3).reset_index(drop=True))
save_df(PIECES, "perfiles_piezas")

def _role(perfil, starts):
    sub = PIECES[(PIECES.perfil == perfil) & PIECES.rol.str.startswith(starts)]
    return sub.iloc[0] if len(sub) else None

srows = []
for perfil, cfg in PERFILES.items():
    a, c, b = _role(perfil, "Segmentador"), _role(perfil, "Clasificador"), _role(perfil, "Integrado")
    ok = lambda x: x is not None and x.estado == "ok"
    params_ac = a.params_M + c.params_M if ok(a) and ok(c) else np.nan
    lat_ac = a.lat_ms + c.lat_ms if ok(a) and ok(c) else np.nan
    srows.append(dict(
        perfil=perfil,
        A=a.exp_id if ok(a) else f"{cfg['A']} (falta)",
        AP50_A=a.AP50 if ok(a) else np.nan, AR100_A=a.AR100 if ok(a) else np.nan,
        C=c.exp_id if ok(c) else f"{cfg['C']} (falta)",
        top1_C=c.top1 if ok(c) else np.nan, macroF1_C=c.macro_f1 if ok(c) else np.nan,
        B=b.exp_id if ok(b) else f"{cfg['B']} (falta)",
        AP50_B=b.AP50 if ok(b) else np.nan, AR100_B=b.AR100 if ok(b) else np.nan,
        AP50_agn_B=b.AP50_agn if ok(b) else np.nan, AR100_agn_B=b.AR100_agn if ok(b) else np.nan,
        acc_clase_B=b.acc_clase if ok(b) else np.nan,
        params_M_A_C=params_ac, lat_ms_A_C=lat_ac,
        params_M_B=b.params_M if ok(b) else np.nan, lat_ms_B=b.lat_ms if ok(b) else np.nan,
        params_M_B_menos_A_C=(b.params_M - params_ac) if ok(b) else np.nan,
        lat_ms_B_menos_A_C=(b.lat_ms - lat_ac) if ok(b) else np.nan,
        brecha_AP50_agn_B_menos_A=(b.AP50_agn - a.AP50) if ok(a) and ok(b) else np.nan,
        brecha_AP50_B_menos_A=(b.AP50 - a.AP50) if ok(a) and ok(b) else np.nan,
        brecha_acc_B_menos_techoC=(b.acc_clase - c.top1) if ok(b) and ok(c) else np.nan,
    ))
SUMM = pd.DataFrame(srows)
print("\nResumen por perfil (AP50/AR100 de máscara, top-1 de clasificación y coste de piezas):")
display(SUMM.round(3).reset_index(drop=True))
save_df(SUMM, "perfiles_resumen")

# Diferencia robusto − ligero en cada pieza (solo donde hay dato en ambos perfiles)
num_cols = ["AP50_A", "AR100_A", "top1_C", "macroF1_C", "AP50_B", "AR100_B",
            "AP50_agn_B", "AR100_agn_B", "acc_clase_B",
            "params_M_A_C", "lat_ms_A_C", "params_M_B", "lat_ms_B"]
if {"ligero", "robusto"} <= set(SUMM.perfil):
    lig = SUMM[SUMM.perfil == "ligero"].iloc[0]
    rob = SUMM[SUMM.perfil == "robusto"].iloc[0]
    delta = {"perfil": "robusto - ligero"}
    for c in num_cols:
        delta[c] = rob[c] - lig[c]
    DELTA = pd.DataFrame([delta])
    print("\nDiferencia robusto - ligero (positivo = el robusto es mayor):")
    display(DELTA.round(3)); save_df(DELTA, "perfiles_delta")

fig, ax = plt.subplots(figsize=(7, 3.8))
x = np.arange(len(SUMM))
width = 0.26
for j, (col, label, color) in enumerate([("AP50_A", "A · AP50", "#94a3b8"),
                                         ("AP50_agn_B", "B · AP50 agnóstico", "#0ea5e9"),
                                         ("top1_C", "C · top-1 (techo)", "#f59e0b")]):
    ax.bar(x + (j - 1) * width, SUMM[col].fillna(0).values, width, label=label, color=color)
ax.set_xticks(x); ax.set_xticklabels(SUMM.perfil)
ax.set_ylim(0, 1); ax.set_ylabel("0–1"); ax.grid(axis="y", alpha=.3)
ax.set_title(f"Perfiles: máscara agnóstica (A y B) y techo de clasificación (C) — {SPLIT}")
ax.legend(fontsize=8)
plt.tight_layout(); save_fig("perfiles_barras"); plt.show()

# Lectura rápida y avisos de comparabilidad
print("\nLectura por perfil:")
for perfil in SUMM.perfil:
    r = SUMM[SUMM.perfil == perfil].iloc[0]
    partes = []
    if pd.notna(r.AP50_A): partes.append(f"A AP50 {r.AP50_A:.3f}")
    if pd.notna(r.top1_C): partes.append(f"C top-1 {r.top1_C:.3f}")
    if pd.notna(r.AP50_agn_B): partes.append(f"B AP50 agn {r.AP50_agn_B:.3f}")
    if pd.notna(r.AP50_B): partes.append(f"B AP50 clases {r.AP50_B:.3f}")
    print(f"  • {perfil}: " + (" | ".join(partes) if partes else "sin datos todavía"))
print("  (A y B agnóstico miden máscara; B con clases añade el error de clasificación; C es techo con máscara real.)")

for perfil in SUMM.perfil:
    a = _role(perfil, "Segmentador"); b = _role(perfil, "Integrado")
    if a is not None and b is not None and pd.notna(a.imgsz) and pd.notna(b.imgsz) and a.imgsz != b.imgsz:
        print(f"  ⚠️ {perfil}: A evalúa a {int(a.imgsz)} px y B a {int(b.imgsz)} px; la latencia y el AP no son directamente comparables.")

# Qué falta exactamente para cerrar los perfiles
PEND = [
    dict(perfil="ligero", pieza="Segmentador A2", archivo="resultados/A2_yolo26n.json",
         cuadernillo="A2_yolo26n_agnostico_corregido.ipynb"),
    dict(perfil="ligero", pieza="Integrado B2", archivo="resultados/B2_yolo26n.json",
         cuadernillo="B2_yolo26n_clases_corregido.ipynb"),
    dict(perfil="extra YOLO26m", pieza="Segmentador A1", archivo="resultados/A1_yolo26m.json",
         cuadernillo="A1_yolo26_agnostico_v2_m.ipynb"),
    dict(perfil="extra YOLO26m", pieza="Integrado B1", archivo="resultados/B1_yolo26m.json",
         cuadernillo="B1_yolo26_clases_v4_final_m.ipynb"),
]
PEND = pd.DataFrame([p for p in PEND if not (WORK / p["archivo"]).exists()])
if PEND.empty:
    print("\nNo falta ningún resultado de los perfiles.")
else:
    print("\nFaltan estos resultados de test (cópialos a la carpeta del proyecto y vuelve a ejecutar):")
    display(PEND.reset_index(drop=True)); save_df(PEND, "perfiles_pendientes")

if SUMM["acc_clase_B"].isna().all():
    print("\n⚠️ B4 no guardó `matched_cls` (exactitud de clase sobre instancias emparejadas): "
          "la comparación con el techo de C queda pendiente de recalcular ese bloque en B4.")


,perfil,rol,exp_id,estado,AP50,AP,AR100,AP50_agn,AR100_agn,top1,top5,macro_f1,acc_clase,det_recall,params_M,lat_ms,imgsz,épocas,piloto
0,ligero,Segmentador A (agnóstico),A2_yolo26n,ok,0.651,0.445,0.535,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.083,15.987,224,40,False
1,ligero,Clasificador C (máscara real),C2_mobilenetv3_large_100,ok,NaN,NaN,NaN,NaN,NaN,0.579,0.840,0.423,NaN,NaN,4.616,10.695,224,15,False
2,ligero,Integrado B (con clases),B2_yolo26n,ok,0.207,0.148,0.236,0.601,0.553,NaN,NaN,NaN,0.674,0.441,3.260,15.765,224,40,False
3,robusto,Segmentador A (agnóstico),A4_mask2former_swin_tiny,ok,0.752,0.548,0.726,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.403,111.286,640,6,False
4,robusto,Clasificador C (máscara real),C3_inception_v4,ok,NaN,NaN,NaN,NaN,NaN,0.578,0.835,0.427,NaN,NaN,41.639,30.553,224,17,False
5,robusto,Integrado B (con clases),B4_mask2former_swin_tiny,ok,0.352,0.265,0.605,0.657,0.709,NaN,NaN,NaN,NaN,NaN,47.486,74.176,224,20,False



Resumen por perfil (AP50/AR100 de máscara, top-1 de clasificación y coste de piezas):


,perfil,A,AP50_A,AR100_A,C,top1_C,macroF1_C,B,AP50_B,AR100_B,...,acc_clase_B,params_M_A_C,lat_ms_A_C,params_M_B,lat_ms_B,params_M_B_menos_A_C,lat_ms_B_menos_A_C,brecha_AP50_agn_B_menos_A,brecha_AP50_B_menos_A,brecha_acc_B_menos_techoC
0,ligero,A2_yolo26n,0.651,0.535,C2_mobilenetv3_large_100,0.579,0.423,B2_yolo26n,0.207,0.236,...,0.674,7.699,26.683,3.260,15.765,-4.438,-10.917,-0.050,-0.444,0.096
1,robusto,A4_mask2former_swin_tiny,0.752,0.726,C3_inception_v4,0.578,0.427,B4_mask2former_swin_tiny,0.352,0.605,...,NaN,89.042,141.839,47.486,74.176,-41.557,-67.662,-0.096,-0.401,NaN



Diferencia robusto - ligero (positivo = el robusto es mayor):


,perfil,AP50_A,AR100_A,top1_C,macroF1_C,AP50_B,AR100_B,AP50_agn_B,AR100_agn_B,acc_clase_B,params_M_A_C,lat_ms_A_C,params_M_B,lat_ms_B
0,robusto - ligero,0.102,0.192,-0.001,0.004,0.145,0.369,0.056,0.156,NaN,81.344,115.156,44.225,58.411



Lectura por perfil:
  • ligero: A AP50 0.651 | C top-1 0.579 | B AP50 agn 0.601 | B AP50 clases 0.207
  • robusto: A AP50 0.752 | C top-1 0.578 | B AP50 agn 0.657 | B AP50 clases 0.352
  (A y B agnóstico miden máscara; B con clases añade el error de clasificación; C es techo con máscara real.)
  ⚠️ robusto: A evalúa a 640 px y B a 224 px; la latencia y el AP no son directamente comparables.

No falta ningún resultado de los perfiles.


## 6. (Opcional) Pipeline P1 = A + C frente a B (modelo integrado)

**P1 = A + C.** El segmentador agnóstico (A) propone máscaras; cada máscara **predicha** se recorta como en el entrenamiento de los clasificadores (caja de la máscara + 2 % de margen, **fondo negro**, lado ≤ 256) y el clasificador (C) la etiqueta; cada instancia emite sus `topk` clases con `score = score_del_segmentador × prob_de_la_clase`. Se evalúa con el mismo AP con clases que B. Así se compara la **propuesta 1** (A+C) con la **propuesta 2** (B, un solo modelo) en igualdad de condiciones.

- La celda siguiente construye los P1 que falten con los A (necesitan `pred_test`) y C (necesitan `best.pt`) que existan; **se omite con `run_pipelines=False`**. Un P1 se compone de dos modelos ya entrenados: no entrena nada.
- Solo se combinan resultados del mismo tipo (reales con reales, pilotos con pilotos).
- La diferencia entre la exactitud de clase de C con máscara *real* (techo) y la de P1 con máscara *predicha* mide **cuánto cuesta la segmentación imperfecta**.
- La latencia de P1 es una **estimación** (latencia de A + n.º medio de recortes × latencia de C por recorte); los parámetros son la suma de ambos.

In [11]:
if CFG["run_pipelines"]:
    # Datos del pipeline: imágenes originales + salida del cuadernillo 00 (sin recortes) + food_utils. Solo se ejecuta con run_pipelines=True.
    ensure('gdown')
    # Rutas explícitas (opcional). None = buscar por nombre en tu Drive.
    DRIVE_PATHS = dict(raw=None, prep=None, utils=None)
    # Ids de los enlaces compartidos (solo se usan si el archivo no aparece en tu Drive montado)
    DRIVE_IDS = dict(raw="17y2oraB_OmagwpnTuiEnBvMPeQlg4nEY", prep="1LmD3j5smb-1NJp4hV0yAagvb7__rT--A", utils="1-9FZCbvzSxJdhq4Z9C8nnbd13Unga8NM")
    PATTERNS  = dict(raw=r"food[-_ ]?recognition[-_ ]?2022.*\.zip$", prep=r"food[-_ ]?prep[-_ ]?v21.*\.zip$", utils=r"^food_utils.*\.py$")
    OUT_NAMES = dict(raw="food-recognition-2022.zip", prep="food-prep-v21.zip", utils="food_utils.py")

    def find_in_drive(kind, max_depth=3):
        pat = re.compile(PATTERNS[kind], re.I)
        roots = [WORK, Path("/content/drive/MyDrive"), Path.cwd()]
        found = []
        for root in roots:
            if not root.exists(): continue
            base = len(root.parts)
            for dp, dn, fn in os.walk(root):
                dn[:] = [d for d in dn if not d.startswith(".") and d not in ("runs", "yolo_data", "predicciones", "resultados", "Trash")]
                if len(Path(dp).parts) - base >= max_depth: dn[:] = []
                for f in fn:
                    if pat.search(f): found.append(Path(dp) / f)
        if not found: return None
        if kind == "utils" and len(found) > 1:
            # varias versiones de food_utils en Drive: se elige la más completa (CropDataset, rotación) y, a igualdad, la más reciente
            def score(p):
                try: t = p.read_text(encoding="utf-8", errors="ignore")
                except Exception: t = ""
                return ("class CropDataset" in t) + ("rotate: float" in t), p.stat().st_mtime
            found.sort(key=score, reverse=True)
            print("   versiones de food_utils en Drive:", [p.name for p in found], "→ se usa", found[0].name)
        return found[0]

    def get_file(kind):
        if DRIVE_PATHS[kind]:
            p = Path(DRIVE_PATHS[kind]); assert p.exists(), f"No existe {p}"; return p
        p = find_in_drive(kind)
        if p: print(f"[{kind}] encontrado en Drive → {p}"); return p
        out = DATA / "zips" / OUT_NAMES[kind]; out.parent.mkdir(parents=True, exist_ok=True)
        print(f"[{kind}] no está en el Drive montado; intento bajarlo con gdown (id {DRIVE_IDS[kind]})...")
        import gdown
        r = gdown.download(id=DRIVE_IDS[kind], output=str(out), quiet=False)
        if not r or not out.exists():
            raise RuntimeError(f"No pude obtener '{kind}'. Copia el archivo a tu Drive (carpeta {WORK}) o indica su ruta en DRIVE_PATHS "
                               f"(gdown falla a menudo en archivos grandes por la cuota de descargas de Drive).")
        return out

    def unzip_once(zpath, dest, exclude=()):
        zpath, dest = Path(zpath), Path(dest)
        marker, sig = dest / ".unzipped_ok", f"{zpath.name}:{zpath.stat().st_size}"
        if marker.exists() and marker.read_text() == sig:
            print(f"   ya extraído en {dest}"); return
        dest.mkdir(parents=True, exist_ok=True)
        free, need = shutil.disk_usage(dest).free, zpath.stat().st_size
        print(f"   zip {need/1e9:.2f} GB | espacio libre {free/1e9:.1f} GB")
        if free < 1.5 * need: print("   ⚠️ poco espacio libre para extraer; si falla, usa un entorno con más disco")
        t0 = time.time()
        if shutil.which("unzip"):
            r = subprocess.run(["unzip", "-q", "-o", str(zpath), "-d", str(dest)] + (["-x", *exclude] if exclude else []))
            if r.returncode > 1: raise RuntimeError(f"unzip falló (código {r.returncode})")
        else:
            import zipfile, fnmatch
            with zipfile.ZipFile(zpath) as z:
                for n in tqdm(z.namelist(), desc="extrayendo"):
                    if not any(fnmatch.fnmatch(n, p) for p in exclude): z.extract(n, dest)
        marker.write_text(sig); print(f"   extraído en {time.time()-t0:.0f} s")

    def find_dir(root, pred, max_depth=5):
        root = Path(root); base = len(root.parts)
        for dp, dn, fn in os.walk(root):
            dn[:] = [d for d in dn if d != "images"]
            if len(Path(dp).parts) - base >= max_depth: dn[:] = []
            if pred(dn, fn): return Path(dp)
        return None

    # 1) imágenes originales
    z_raw = get_file("raw");  print("→ extrayendo imágenes originales"); unzip_once(z_raw, DATA / "raw")
    RAW_DIR = find_dir(DATA / "raw", lambda dn, fn: "public_training_set_release_2.0" in dn)
    assert RAW_DIR is not None, "El zip original no contiene la carpeta public_training_set_release_2.0"
    for k, sub in (("train", "public_training_set_release_2.0"), ("val", "public_validation_set_2.0")):
        d = RAW_DIR / sub / "images"; assert d.exists(), f"Falta {d}"
        print(f"   imágenes {k}: {sum(1 for _ in os.scandir(d)):,}")

    # 2) salida del cuadernillo 00 (se omiten los recortes: A4/B4 no los usan)
    z_prep = get_file("prep"); print("→ extrayendo food-prep-v21 (sin recortes)"); unzip_once(z_prep, DATA / "prep", exclude=("crops/*", "*/crops/*", "crops_ctx/*", "*/crops_ctx/*"))
    PREP_DIR = find_dir(DATA / "prep", lambda dn, fn: "classes.json" in fn and "splits.json" in fn)
    assert PREP_DIR is not None, "El zip de prep no contiene classes.json + splits.json"
    missing = [f for f in ("manifest.json", "coco_train.json", "coco_val.json", "coco_test.json", "coco_agnostic_train.json") if not (PREP_DIR / f).exists()]
    assert not missing, f"Faltan archivos en {PREP_DIR}: {missing}"

    # 3) food_utils
    u = get_file("utils"); CODE = DATA / "code"; CODE.mkdir(exist_ok=True); shutil.copy(u, CODE / "food_utils.py")
    sys.path.insert(0, str(CODE)); sys.modules.pop("food_utils", None)
    print("\nRAW_DIR :", RAW_DIR, "\nPREP_DIR:", PREP_DIR, "\nutils   :", CODE / "food_utils.py")
else:
    print("Sección omitida (CFG['run_pipelines']=False). Los P1 que ya existan se comparan más abajo.")

Sección omitida (CFG['run_pipelines']=False). Los P1 que ya existan se comparan más abajo.


In [12]:
if CFG["run_pipelines"]:
    import torch, torch.nn as nn, torch.nn.functional as F, cv2
    from PIL import Image
    ensure("pycocotools")
    def ensure_timm(min_version="1.0.20"):
        try:
            import timm
            from packaging.version import Version
            assert Version(timm.__version__) >= Version(min_version)
        except Exception:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "timm"], check=True); importlib.invalidate_caches()
    ensure_timm()
    CUDA = torch.cuda.is_available(); DEVICE = "cuda" if CUDA else "cpu"
    AMP_DTYPE = torch.bfloat16 if (CUDA and torch.cuda.is_bf16_supported()) else torch.float16
    if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))
    sys.modules.pop("food_utils", None)
    import food_utils as fu
    prep = fu.Prep(str(PREP_DIR), raw_dir=str(RAW_DIR)); K = prep.K
    BAD = {s: prep.bad_image_ids(s) for s in ("val", "test")}
    INFO_BY_ID = {im["id"]: im for im in prep.images("test")}
    print("food_utils", fu.__version__, "| clases:", K, "| dispositivo:", DEVICE)

    def masked_crop(img, mask, sx, sy, margin=None, max_side=None, erode=None):
        """Recorte como el del cuadernillo 00: caja de la máscara + margen, reducción a lado <= max_side y fondo NEGRO fuera de la máscara."""
        margin = CFG["crop_margin"] if margin is None else margin; max_side = max_side or CFG["crop_max_side"]; erode = CFG["mask_erode_px"] if erode is None else erode
        bb = fu.mask_bbox_array(mask)                                     # [x, y, w, h] en el marco COCO
        if bb is None: return None
        W, H = img.size
        x0, y0, x1, y1 = bb[0] * sx, bb[1] * sy, (bb[0] + bb[2]) * sx, (bb[1] + bb[3]) * sy
        mx, my = (x1 - x0) * margin, (y1 - y0) * margin
        box = (max(0, int(x0 - mx)), max(0, int(y0 - my)), min(W, int(np.ceil(x1 + mx))), min(H, int(np.ceil(y1 + my))))
        if box[2] - box[0] < 2 or box[3] - box[1] < 2: return None
        sub = img.crop(box)
        mk = mask[int(box[1] / sy):int(np.ceil(box[3] / sy)), int(box[0] / sx):int(np.ceil(box[2] / sx))]
        if mk.size == 0: return None
        mk = cv2.resize(mk.astype(np.uint8), sub.size, interpolation=cv2.INTER_NEAREST) > 0
        s = min(1.0, max_side / max(sub.size)); sz = (max(1, round(sub.size[0] * s)), max(1, round(sub.size[1] * s)))
        if sz != sub.size:
            sub = sub.resize(sz, Image.LANCZOS); mk = cv2.resize(mk.astype(np.uint8) * 255, sz, interpolation=cv2.INTER_AREA) > 127
        if erode > 0:
            e = cv2.erode(mk.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * erode + 1, 2 * erode + 1))) > 0
            if e.any(): mk = e
        arr = np.array(sub); arr[~mk] = 0
        return Image.fromarray(arr)

    class SE(nn.Module):
        def __init__(self, c, r=16):
            super().__init__(); h = max(8, c // r)
            self.fc1, self.fc2 = nn.Conv2d(c, h, 1), nn.Conv2d(h, c, 1)
        def forward(self, x):
            s = x.mean((2, 3), keepdim=True)
            return x * torch.sigmoid(self.fc2(F.relu(self.fc1(s))))

    class Block(nn.Module):
        def __init__(self, cin, cout, stride, drop_path=0.0):
            super().__init__()
            self.c1, self.b1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False), nn.BatchNorm2d(cout)
            self.c2, self.b2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout)
            self.se = SE(cout)
            self.short = None if (stride == 1 and cin == cout) else nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
            self.drop_path = drop_path
        def forward(self, x):
            y = self.se(self.b2(self.c2(F.relu(self.b1(self.c1(x)), inplace=True))))
            return F.relu(y + (x if self.short is None else self.short(x)), inplace=True)

    class FoodNet(nn.Module):          # C1 (misma arquitectura que el cuadernillo C1)
        def __init__(self, num_classes, widths=(64, 128, 256, 512), blocks=(2, 2, 2, 2), dropout=0.3, drop_path=0.0):
            super().__init__()
            self.stem = nn.Sequential(nn.Conv2d(3, 32, 3, 2, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
                                      nn.Conv2d(32, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
            total, k, cin, layers = sum(blocks), 0, widths[0], []
            for i, (w, n) in enumerate(zip(widths, blocks)):
                for j in range(n):
                    layers.append(Block(cin, w, 2 if (j == 0 and i > 0) else 1, 0.0)); cin = w; k += 1
            self.stages = nn.Sequential(*layers)
            self.drop, self.fc = nn.Dropout(dropout), nn.Linear(cin, num_classes)
        def forward(self, x):
            return self.fc(self.drop(self.stages(self.stem(x)).mean((2, 3))))

    class TimmClassifier(nn.Module):   # C2–C4 (misma cabeza que los cuadernillos C2–C4)
        def __init__(self, backbone, feat_dim, num_classes, dropout):
            super().__init__()
            self.backbone, self.drop, self.fc = backbone, nn.Dropout(dropout), nn.Linear(feat_dim, num_classes)
        def forward(self, x):
            f = self.backbone(x)
            if f.ndim == 4: f = f.mean((2, 3))
            elif f.ndim == 3: f = f.mean(1)
            return self.fc(self.drop(f))

    def _load_ckpt(path):
        try: return torch.load(path, map_location="cpu", weights_only=False)
        except TypeError: return torch.load(path, map_location="cpu")

    def find_weights(rec):
        c = []
        w = _get(rec, "files", "weights")
        if w: c += [Path(w), WORK / "runs" / Path(w).parent.name / Path(w).name]
        c += [WORK / "runs" / rec["exp_id"] / "best.pt"]
        return next((p for p in c if p.exists()), None)

    def find_preds(rec, split="test"):
        c = []
        p = _get(rec, "files", "pred_" + split)
        if p: c += [Path(p), WORK / "predicciones" / Path(p).name, WORK / "predicciones_pilot" / Path(p).name]
        c += [WORK / "predicciones" / f"{rec['exp_id']}_{split}.json", WORK / "predicciones_pilot" / f"{rec['exp_id']}_{split}.json"]
        return next((x for x in c if x.exists()), None)

    class ClsModel:
        """Reconstruye un clasificador C desde su resultado (cfg + pesos) y lo aplica a recortes PIL con el MISMO preprocesado del entrenamiento."""
        def __init__(self, rec):
            cfg, tr = rec["cfg"], rec.get("train") or {}
            self.size = int(cfg["imgsz"]); norm = tr.get("pretrained_norm") or {}
            self.mean, self.std = tuple(norm.get("mean") or fu.IMAGENET_MEAN), tuple(norm.get("std") or fu.IMAGENET_STD)
            if cfg.get("arch"):
                import timm
                bb = None
                for kw in (dict(num_classes=0, global_pool="avg"), dict(num_classes=0)):
                    try: bb = timm.create_model(cfg["arch"], pretrained=False, **kw); break
                    except TypeError: continue
                with torch.no_grad():
                    bb.eval(); f = bb(torch.zeros(1, 3, self.size, self.size))
                model = TimmClassifier(bb, f.shape[1] if f.ndim == 4 else f.shape[-1], K, cfg.get("dropout", 0.0))
            else:
                model = FoodNet(K, tuple(cfg["widths"]), tuple(cfg["blocks"]), cfg.get("dropout", 0.0), 0.0)
            model.load_state_dict(_load_ckpt(find_weights(rec))["model"])
            self.model = model.to(DEVICE).eval().to(memory_format=torch.channels_last)
        @torch.no_grad()
        def predict_pil(self, crops):
            X = np.stack([fu.crop_to_array(fu.square_letterbox(c, self.size), self.mean, self.std) for c in crops])
            out = []
            for i in range(0, len(X), CFG["cls_batch"]):
                x = torch.from_numpy(X[i:i + CFG["cls_batch"]]).to(DEVICE).to(memory_format=torch.channels_last)
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=CUDA): lg = self.model(x)
                out.append(lg.float().softmax(-1).cpu().numpy())
            return np.concatenate(out)
else:
    print("Sección omitida (CFG['run_pipelines']=False). Los P1 que ya existan se comparan más abajo.")

Sección omitida (CFG['run_pipelines']=False). Los P1 que ya existan se comparan más abajo.


In [13]:
if CFG["run_pipelines"]:
    def classification_on_matches(split, dets, ids, score_thr, iou_thr=0.5):
        """Empareja cada GT con la detección de mayor IoU (≥ iou_thr, sin mirar la clase) y compara las clases."""
        from pycocotools import mask as mu
        g = fu._gt_dict(prep.coco(split)); idset = set(ids)
        gd = dict(images=[im for im in g["images"] if im["id"] in idset],
                  annotations=[a for a in g["annotations"] if a["image_id"] in idset], categories=g["categories"])
        gtc = fu._make_coco(gd)
        cat_ids = sorted(c["id"] for c in g["categories"]); idx = {c: i for i, c in enumerate(cat_ids)}
        nm = {c["id"]: c["name"] for c in g["categories"]}; names = [nm[c] for c in cat_ids]
        gts = defaultdict(list)
        for a in gd["annotations"]:
            if not a.get("iscrowd", 0): gts[a["image_id"]].append(a)
        dts = defaultdict(list)
        for d in dets:
            if d["image_id"] in idset and d["score"] >= score_thr: dts[d["image_id"]].append(d)
        y_true, y_pred = [], []
        n_gt = sum(len(v) for v in gts.values()); n_det = sum(len(v) for v in dts.values())
        for iid, gl in gts.items():
            dl = sorted(dts.get(iid, []), key=lambda d: -d["score"])
            if not dl: continue
            iou = np.asarray(mu.iou([d["segmentation"] for d in dl], [gtc.annToRLE(a) for a in gl], [0] * len(gl)))
            used = set()
            for i, d in enumerate(dl):
                for j in np.argsort(-iou[i]):
                    if iou[i, j] < iou_thr: break
                    if j in used: continue
                    used.add(int(j)); y_true.append(idx[gl[j]["category_id"]]); y_pred.append(idx[d["category_id"]]); break
        out = dict(score_thr=score_thr, iou_thr=iou_thr, n_gt=n_gt, n_det=n_det, n_matched=len(y_true),
                   det_recall=len(y_true) / max(n_gt, 1), det_precision=len(y_true) / max(n_det, 1))
        if y_true:
            Kc = len(names); P = np.zeros((len(y_true), Kc), dtype=np.float32); P[np.arange(len(y_true)), y_pred] = 1.0
            cm = fu.classification_metrics(y_true, P, names=names, topk=(1,))
            out.update(accuracy=cm["top1"], macro_f1=cm["macro_f1"], balanced_acc=cm["balanced_acc"])
        return out

    A_LIST = [e for e, r in R.items() if r["family"] == "A" and find_preds(r)]
    C_LIST = [e for e, r in R.items() if r["family"] == "C" and find_weights(r)]
    TODO = [(a, c) for a in A_LIST for c in C_LIST if R[a]["_pilot"] == R[c]["_pilot"] and (CFG["rebuild_pipelines"] or f"P1_{a}_{c}" not in ALL)]
    print(f"Segmentadores A utilizables: {A_LIST}\nClasificadores C utilizables: {C_LIST}")
    skipped = [f"P1_{a}_{c}" for a in A_LIST for c in C_LIST if R[a]["_pilot"] == R[c]["_pilot"] and f"P1_{a}_{c}" in ALL and not CFG["rebuild_pipelines"]]
    if skipped: print("Ya calculados (se reutilizan):", skipped)
    if not A_LIST or not C_LIST: print("⚠️ Hace falta al menos un A con predicciones y un C con pesos para construir P1.")
    print(f"Pipelines por calcular: {len(TODO)}")

    CLS_MODELS = {}
    for a in sorted({a for a, _ in TODO}):
        cs = [c for aa, c in TODO if aa == a]
        for c in cs:
            if c not in CLS_MODELS:
                try: CLS_MODELS[c] = ClsModel(R[c]); print("  clasificador cargado:", c, f"(imgsz {CLS_MODELS[c].size})")
                except Exception as e: print(f"  ⚠️ no se pudo cargar {c}: {type(e).__name__}: {e}")
        cs = [c for c in cs if c in CLS_MODELS]
        if not cs: continue
        t0 = time.time(); recA = R[a]; pilot = recA["_pilot"]
        dets = fu.load_predictions(find_preds(recA))
        n_img = _get(recA, "metrics", "test", "agnostic", "n_images")
        infos = [im for im in prep.images("test") if im["id"] not in BAD["test"]]
        if n_img: infos = infos[:n_img]
        if CFG["pipeline_limit_images"]: infos = infos[:CFG["pipeline_limit_images"]]
        ids = [im["id"] for im in infos]; idset = set(ids)
        by_img = defaultdict(list)
        for d in dets:
            if d["image_id"] in idset and d["score"] >= CFG["det_min_score"]: by_img[d["image_id"]].append(d)
        dets_f = [d for iid in ids for d in sorted(by_img.get(iid, []), key=lambda d: -d["score"])[:CFG["max_det_per_img"]]]
        probs = {c: np.zeros((len(dets_f), K), np.float16) for c in cs}; has = np.zeros(len(dets_f), bool)
        idx_by = defaultdict(list)
        for i, d in enumerate(dets_f): idx_by[d["image_id"]].append(i)
        for iid, idxs in tqdm(idx_by.items(), desc=f"{a}: recortes + clasificación"):
            info = INFO_BY_ID[iid]
            try: img, mode, sx, sy = fu.load_image_oriented(prep.image_path(info), info)
            except Exception: continue
            if mode == "swapped": continue
            valid, crops = [], []
            for i in idxs:
                cr = masked_crop(img, fu.rle_decode(dets_f[i]["segmentation"]), sx, sy)
                if cr is not None: valid.append(i); crops.append(cr)
            if not valid: continue
            has[valid] = True
            for c in cs: probs[c][valid] = CLS_MODELS[c].predict_pil(crops)
        keep = np.where(has)[0]; dets_v = [dets_f[i] for i in keep]
        print(f"[{a}] {len(ids)} fotos | detecciones clasificadas: {len(dets_v):,} de {len(dets_f):,} | {time.time()-t0:.0f} s")
        for c in cs:
            pv = probs[c][keep].astype(np.float32)
            dets_b = fu.combine_with_classifier(dets_v, pv, topk=CFG["topk"], score_mode="product")
            EV = fu.evaluate_all(prep.coco("test"), dets_b, img_ids=ids, per_class=True, n_folds=CFG["eval_folds"])
            top1 = [dict(d, category_id=int(p.argmax()) + 1) for d, p in zip(dets_v, pv)]
            EV["matched_cls"] = classification_on_matches("test", top1, ids, CFG["match_score_thr"])
            recC = R[c]; eA, eC = recA.get("efficiency") or {}, recC.get("efficiency") or {}
            latA, latC = _num(_get(eA, "latency", "mean_ms")), _num(_get(eC, "latency", "mean_ms"))
            n_avg = len(dets_v) / max(1, len(ids))
            eff = dict(params=_num(eA.get("params")) + _num(eC.get("params")), params_A=eA.get("params"), params_C=eC.get("params"),
                       latency=dict(mean_ms=latA + n_avg * latC, note="estimada = latencia A + recortes medios por foto x latencia C"),
                       latency_A_ms=latA, latency_C_ms_per_crop=latC, crops_per_image=n_avg, gpu_mem_mb=max(_num(eA.get("gpu_mem_mb")), _num(eC.get("gpu_mem_mb"))))
            eid = f"P1_{a}_{c}"
            path = fu.write_result(
                eid, model=f"{recA.get('model')} + {recC.get('model')}", metrics={"test": EV}, prep=prep, seed=CFG["seed"], efficiency=eff,
                cfg=dict(A=a, C=c, pilot=pilot, det_min_score=CFG["det_min_score"], max_det_per_img=CFG["max_det_per_img"], topk=CFG["topk"],
                         score_mode="product", crop_margin=CFG["crop_margin"], imgsz_A=_get(recA, "cfg", "imgsz"), imgsz_C=_get(recC, "cfg", "imgsz")),
                train=dict(pilot=pilot, note="sin entrenamiento propio: compone un A y un C ya entrenados"),
                files=dict(A_pred=str(find_preds(recA)), A_weights=_get(recA, "files", "weights"), C_weights=str(find_weights(recC))),
                notes="Pipeline P1: segmentador agnostico + clasificador sobre recortes de MASCARAS PREDICHAS (fondo negro, margen 2 %).",
                out_dir=WORK / ("resultados_pilot" if pilot else "resultados"))
            m = EV["matched_cls"]
            print(f"   {eid}: AP50 {EV['class_aware'].get('AP50')} | AR100 {EV['class_aware'].get('AR100')} | exactitud de clase (emparejadas) {m.get('accuracy')} | → {path.name}")
        if CUDA: torch.cuda.empty_cache()
    print("\nListo. Las celdas siguientes releen los resultados.")
else:
    print("Sección omitida (CFG['run_pipelines']=False). Los P1 que ya existan se comparan más abajo.")

Sección omitida (CFG['run_pipelines']=False). Los P1 que ya existan se comparan más abajo.


In [14]:
ALL, BAD_FILES = read_results(RESULT_DIRS); R = select_results(ALL, verbose=False)
PIPE = seg_cls_table(("P",)); BMOD = seg_cls_table(("B",))
if PIPE.empty:
    print("Todavía no hay pipelines P1 (A+C)." + ("" if CFG["run_pipelines"] else " Para construirlos, pon CFG['run_pipelines']=True y ejecuta la sección 6."))
    if not BMOD.empty: print("Mientras tanto, la propuesta 2 (B) está en la sección 3.")
else:
    both = pd.concat([PIPE.assign(propuesta="P1 (A+C)"), BMOD.assign(propuesta="P2 = B (integrado)")], ignore_index=True)
    cols = ["propuesta", "exp_id", "AP50", "AR100", "AP", "AP50_agn", "det_recall", "acc_clase", "macroF1_clase", "params_M", "lat_ms", "piloto"]
    display(both.sort_values("AP50", ascending=False)[cols].round(3).reset_index(drop=True)); save_df(both, "pipelines_vs_B")
    f = bar_compare(both.sort_values("AP50", ascending=False), ["AP50", "AR100", "acc_clase"], f"P1 (A+C) frente a B ({SPLIT})")
    if f is not None: save_fig("pipelines_vs_B"); plt.show()
    # ¿cuánto cuesta segmentar mal? exactitud de clase con máscara real (C) frente a máscara predicha (P1) y modelo integrado (B)
    rows = []
    for eid, r in R.items():
        if r["family"] != "P": continue
        c = _get(r, "cfg", "C"); top1_c = _num(_get(R.get(c), "metrics", SPLIT, "top1")) if c in R else np.nan
        rows.append(dict(pipeline=eid, C=c, top1_C_máscara_real=top1_c, acc_P1_máscara_predicha=_num(_get(r, "metrics", SPLIT, "matched_cls", "accuracy")),
                         recall_detección=_num(_get(r, "metrics", SPLIT, "matched_cls", "det_recall"))))
    if rows:
        D = pd.DataFrame(rows); D["coste_segmentación"] = D.top1_C_máscara_real - D.acc_P1_máscara_predicha
        for _, b in BMOD.iterrows(): D.loc[len(D)] = dict(pipeline=b.exp_id + " (B)", C=None, top1_C_máscara_real=np.nan, acc_P1_máscara_predicha=b.acc_clase, recall_detección=b.det_recall)
        print("\nExactitud de clase: máscara REAL (techo de C) vs máscara PREDICHA (P1) vs modelo integrado (B)")
        display(D.round(3)); save_df(D, "coste_segmentacion")
        print("coste_segmentación = top-1 de C con máscara real − exactitud de P1 sobre las instancias emparejadas. "
              "No son poblaciones idénticas (C evalúa todas las instancias recortables; P1 solo las que el segmentador encontró), así que es una guía, no una resta exacta.")

Todavía no hay pipelines P1 (A+C). Para construirlos, pon CFG['run_pipelines']=True y ejecuta la sección 6.
Mientras tanto, la propuesta 2 (B) está en la sección 3.


## 7. Resumen y exportación

Una sola tabla con todos los experimentos disponibles (cifras de **test**), más la lista de lo que falta y de qué comparaciones quedan limitadas.

In [15]:
ALL, BAD_FILES = read_results(RESULT_DIRS); R = select_results(ALL, verbose=False)     # relee: incluye los P1 recién calculados
rows = []
for eid, r in sorted(R.items()):
    fam, m = r["family"], _get(r, "metrics", SPLIT) or {}
    d = dict(familia=fam, exp_id=eid, modelo=r.get("model"), piloto=r["_pilot"])
    if fam == "A":
        d.update(métrica_principal="AP50 (agnóstico)", valor=_num(_get(m, "agnostic", "AP50")), AR100=_num(_get(m, "agnostic", "AR100")), AP=_num(_get(m, "agnostic", "AP")))
    elif fam in ("B", "P"):
        d.update(métrica_principal="AP50 (con clases)", valor=_num(_get(m, "class_aware", "AP50")), AR100=_num(_get(m, "class_aware", "AR100")), AP=_num(_get(m, "class_aware", "AP")),
                 acc_clase=_num(_get(m, "matched_cls", "accuracy")), det_recall=_num(_get(m, "matched_cls", "det_recall")))
    elif fam == "C":
        d.update(métrica_principal="top-1 (máscara real)", valor=_num(m.get("top1")), macro_f1=_num(m.get("macro_f1")), top5=_num(m.get("top5")))
    eff = r.get("efficiency") or {}
    d.update(params_M=_num(eff.get("params")) / 1e6, lat_ms=_num(_get(eff, "latency", "mean_ms")))
    rows.append(d)
SUMMARY = pd.DataFrame(rows)
if SUMMARY.empty:
    print("Sin resultados que resumir.")
else:
    display(SUMMARY.round(3)); save_df(SUMMARY, "resumen")
    try: SUMMARY.round(4).to_excel(OUT / "resumen.xlsx", index=False)
    except Exception as e: print("(sin XLSX:", type(e).__name__, ")")
    (OUT / "resumen.json").write_text(json.dumps(SUMMARY.where(SUMMARY.notna(), None).to_dict("records"), indent=1, ensure_ascii=False, default=str), encoding="utf-8")

# ---------- qué se pudo y qué no ----------
fams = {f: [e for e, r in R.items() if r["family"] == f] for f in "ABCP"}
lines = []
lines.append("• Segmentadores agnósticos (A): " + (f"{len(fams['A'])} → " + ", ".join(sorted(fams['A'])) if fams['A'] else "ninguno"))
lines.append("• Segmentadores con clases (B): " + (f"{len(fams['B'])} → " + ", ".join(sorted(fams['B'])) if fams['B'] else "ninguno"))
lines.append("• Clasificadores (C): " + (f"{len(fams['C'])} → " + ", ".join(sorted(fams['C'])) if fams['C'] else "ninguno"))
lines.append("• Pipelines P1 (A+C): " + (f"{len(fams['P'])}" if fams['P'] else "ninguno todavía"))
print("\n".join(lines))
lim = []
if len(fams["A"]) < 2: lim.append("no se puede comparar segmentadores agnósticos entre sí (hace falta más de un A)")
if len(fams["C"]) < 2: lim.append("no se puede comparar clasificadores entre sí (hace falta más de un C)")
if not fams["B"]: lim.append("falta la propuesta 2 (B): no hay con qué comparar el pipeline")
if fams["A"] and fams["C"] and not fams["P"]: lim.append("hay A y C pero no P1: ejecuta la sección 6 con run_pipelines=True para medir el pipeline completo")
if not fams["A"] or not fams["C"]: lim.append("sin A o sin C no se puede construir el pipeline P1")
if any(r["_pilot"] for r in R.values()): lim.append("hay resultados de piloto: no los uses en el informe")
print("\nLimitaciones de esta comparación:" + ("".join(f"\n  - {x}" for x in lim) if lim else " ninguna"))
print("\nArchivos en:", OUT)

,familia,exp_id,modelo,piloto,métrica_principal,valor,AR100,AP,params_M,lat_ms,acc_clase,det_recall,macro_f1,top5
0,A,A1_yolo26m,YOLO26m-seg,False,AP50 (agnóstico),0.682,0.551,0.475,26.970,20.803,NaN,NaN,NaN,NaN
1,A,A2_yolo26n,YOLO26n-seg,False,AP50 (agnóstico),0.651,0.535,0.445,3.083,15.987,NaN,NaN,NaN,NaN
2,A,A4_mask2former_swin_tiny,Mask2Former Swin-tiny,False,AP50 (agnóstico),0.752,0.726,0.548,47.403,111.286,NaN,NaN,NaN,NaN
3,B,B1_yolo26m,YOLO26m-seg,False,AP50 (con clases),0.325,0.326,0.232,27.549,19.869,0.736,0.585,NaN,NaN
4,B,B2_yolo26n,YOLO26n-seg,False,AP50 (con clases),0.207,0.236,0.148,3.260,15.765,0.674,0.441,NaN,NaN
5,B,B4_mask2former_swin_tiny,Mask2Former Swin-tiny,False,AP50 (con clases),0.352,0.605,0.265,47.486,74.176,NaN,NaN,NaN,NaN
6,C,C1_cnn_propia,"FoodNet (CNN propia, desde cero)",False,top-1 (máscara real),0.506,NaN,NaN,11.442,6.875,NaN,NaN,0.308,0.776
7,C,C2_mobilenetv3_large_100,timm mobilenetv3_large_100.ra4_e3600_r224_in1k,False,top-1 (máscara real),0.579,NaN,NaN,4.616,10.695,NaN,NaN,0.423,0.840
8,C,C3_inception_v4,timm inception_v4.tf_in1k,False,top-1 (máscara real),0.578,NaN,NaN,41.639,30.553,NaN,NaN,0.427,0.835


• Segmentadores agnósticos (A): 3 → A1_yolo26m, A2_yolo26n, A4_mask2former_swin_tiny
• Segmentadores con clases (B): 3 → B1_yolo26m, B2_yolo26n, B4_mask2former_swin_tiny
• Clasificadores (C): 3 → C1_cnn_propia, C2_mobilenetv3_large_100, C3_inception_v4
• Pipelines P1 (A+C): ninguno todavía

Limitaciones de esta comparación:
  - hay A y C pero no P1: ejecuta la sección 6 con run_pipelines=True para medir el pipeline completo

Archivos en: calorie_counter\comparacion


---
### Cómo interpretar y qué hacer después
- **Primero mira la cobertura** (secciones 1 y 1.1): lo que falta o está incompleto no se puede comparar; el resto sí.
- **Segmentadores:** ordena por AP50 y AR100 (¿encuentra los alimentos?) y usa AP75/AP solo como desempate; con máscaras originales toscas como referencia, el borde fino no se puede juzgar con estas cifras.
- **Clasificadores:** el top-1 con máscara real es un **techo**. Compara siempre en la misma resolución y mira los intervalos y la comparación pareada antes de declarar un ganador.
- **Perfiles (sección 5):** compara el perfil ligero (A2+C2 / B2) con el robusto (A4+C3 / B4) mirando tres cosas: máscara (AP50/AR100 de A), techo de clasificación (top-1 de C) y resultado integrado (AP50 de B). La suma de parámetros y latencias de A+C es el coste de las piezas, no una latencia end-to-end.
- **Propuesta 1 frente a 2:** la comparación estricta es P1 (A+C con máscaras predichas) frente a B; mientras no se construya P1, la sección 5 compara las piezas y deja claro qué es techo y qué es resultado real.
- **Resultados incompletos:** vuelve a ejecutar este cuadernillo cuando lleguen más JSON; no necesita reentrenar nada.
- Los pilotos solo comprueban que el flujo funciona; no los cites.